In [ ]:
!pip install numpy==2.2.6 \
             torch==2.6.0 \
             torchvision==0.21.0 \
             nltk==3.9.4 \
             spacy==3.8.14 \
             matplotlib==3.10.9 \
             tqdm==4.67.3 \
             transformers==5.9.0 \
             datasets==4.8.5 \
             scikit-learn==1.7.2 
!python -m spacy download en_core_web_sm

## Халюцинации
Ти си изследовател в лаборатория за изкуствен интелект и ръководиш важен проект за създаване на мащабен мултимодален набор от данни. За да автоматизирате и ускорите процеса, екипът ти е използвал Голям езиков модел (LLM), който да генерира текстови описания (captions) за стотици хиляди изображения.

За съжаление, при анотиране на данни от LLM често се получават сериозни отклонения. Вследствие на дефекти в декодирането или липса на достатъчно визуален контекст, моделът понякога започва да "халюцинира" обекти, които изобщо не съществуват на снимката (например, твърди, че вижда куче, когато на нея има котка), или генерира напълно несвързан и нелогичен текст (incoherent text). Твоята задача е да изчистиш този набор от данни, като създадеш стабилен филтър, който да разпознава дали дадено описание е коректно, халюцинирано или напълно несвързано.
### Задача и ограничения
Системата, която разработваш, ще бъде интегрирана в среда с изключително строги лимити за памет и изчислително време. Затова главният инженер на проекта е наложил следните железни правила за твоето решение:
- Фиксиран класификатор: Задължително трябва да използваш предварително зададения **GradientBoostingClassifier**. Абсолютно е забранено да променяш неговите хиперпараметри или да добавяш други модели (като невронни мрежи) към пайплайна за класификация.

- Брой характеристики: Имаш право да конструираш и подадеш максимум **16 features** (характеристики) към класификатора. Ще трябва внимателно да подбереш кои метрики носят най-много семантична и визуална информация.

- Без обучение: Позволяват се само методи **без контролирано обучение (unsupervised learning only allowed)** с етикетите на **примерите от данните** за конструиране на характеристиките

- Библиотеки: За извличането на тези характеристики можеш да използваш единствено:
    - `transfomers` – за извличане на ембединги и пресмятане на семантичната близост между изображението и текста, но само и единствено зареденият модел.
    - `nltk` – като е позволено само използването на изтегления модул `punkt` за токенизация.
    - `torch`и `numpy` - За обработка на ембедингите, но не и за трениране
    - `spacy` – за по-дълбок лингвистичен анализ (import spacy и зареждане на модела чрез `nlp = spacy.load("en_core_web_sm")`).

Никакви други външни библиотеки не са позволени!

### Данни
Наборът от данни се зарежда чрез библиотеката `datasets` (чрез `load_from_disk`) и съдържа следните ключови полета за всеки пример:
- `image`: Изображението (PIL Image).
- `caption`: Текстовото описание, генерирано от LLM.
- `label`: Целевият клас, който трябва да предскажеш (0, 1 или 2).

Етикетите отговарят на трите възможни състояния на текста:
- Клас 0: Коректно описание.
- Клас 1: Халюцинация (подменено съществително).
- Клас 2: Несвързан текст (разбъркани думи).

За твоите експерименти разполагаш с тренировъчен (train) и валидационен (val) сплит. Препоръчително е да разгледаш суровите данни, за да придобиеш интуиция за разликите между класовете, преди да започнеш с извличането на характеристиките.
### Налични инструменти

За да те улеснят поне малко в задачата, старшите изследователи от екипа са ти подготвили помощна функция. Тя използва spacy и автоматично връща частите на речта (Part-of-Speech tags) на думите в подаденото изречение.

Твоята цел е креативно да комбинираш тази лингвистична информация (структура на изречението, брой съществителни, граматическа коректност) с визуалното разбиране на CLIP (дали текстът реално отговаря на снимката), за да "събереш" перфектните 64 характеристики и да спасиш данните на лабораторията!
### Оценяване

Оценяването се изпълнява на скрит набор от данни (test split), а водещата метрика за класиране на моделите е F1 macro, за да се гарантира еднакво добро разпознаване и на трите класа.

In [2]:
import numpy as np
import torch
import nltk
import spacy
import matplotlib.pyplot as plt

from tqdm import tqdm
from transformers import CLIPProcessor, CLIPModel
from datasets import load_from_disk

from sklearn.ensemble import GradientBoostingClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    accuracy_score,
    f1_score,
)

/Users/delyan-boychev/.local/share/mamba/envs/ioai/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [ ]:
# 1. NLTK Setup
print("Изтегляне на NLTK ресурси (punkt)...")
nltk.download("punkt", quiet=True)
nltk.download("punkt_tab", quiet=True)

# 2. spaCy Setup
print("Зареждане на spaCy (en_core_web_sm)...")
nlp = spacy.load("en_core_web_sm")

# 3. Зареждане на данните (Локално)
print("Зареждане на тренировъчните и валидационните данни...")
labeled_data = load_from_disk("./alignment_competition_data/labeled")
print(f"Успешно заредени сплитове: {list(labeled_data.keys())}")

# 4. CLIP Setup
print("Инициализация на CLIP модела...")
if torch.cuda.is_available():
    device = "cuda"
elif torch.backends.mps.is_available():
    device = "mps"
else:
    device = "cpu"

print(f"Използвано устройство (Device): {device.upper()}")

model_id = "openai/clip-vit-base-patch32"
processor = CLIPProcessor.from_pretrained(model_id)
model = CLIPModel.from_pretrained(model_id).to(device)
model.eval() 

print("Всички ресурси и данни са заредени успешно!")

Изтегляне на NLTK ресурси (punkt)...
Зареждане на spaCy (en_core_web_sm)...
Зареждане на тренировъчните и валидационните данни...
Успешно заредени сплитове: ['train', 'val']
Инициализация на CLIP модела...
Използвано устройство (Device): MPS


Loading weights: 100%|██████████| 398/398 [00:00<00:00, 53352.07it/s]


Всички ресурси и данни са заредени успешно!


### Визуализация

In [ ]:
# Помощна функция, предоставена от старшите изследователи :)
def get_pos_tags(text):
    """
    Анализира текст и връща списък с думите и техните части на речта.
    Използва вече заредения spaCy модел (nlp) от предходната клетка.
    """
    doc = nlp(text)
    # Генерираме четим формат: "дума (ЧАСТ_НА_РЕЧТА)"
    return [f"{token.text} ({token.pos_})" for token in doc]

# Речник за по-лесно разчитане на класовете
label_map = {
    0: "0: Оригинал (Коректно)",
    1: "1: Сменен обект (Халюцинация)",
    2: "2: Разбъркани думи (Несвързано)"
}

# Визуализираме точно 2 примера
for i in [0, 810]:
    sample = labeled_data["train"][i]
    image = sample["image"]
    caption = sample["caption"]
    label = sample["label"]
    
    print("\n" + "="*60)
    print(f"Пример {i+1}")
    print(f"Етикет (Label): {label_map.get(label, 'Неизвестен')}")
    print(f"Текст (Caption): {caption}")
    print("-" * 60)
    
    # Извикваме помощната функция
    pos_tags = get_pos_tags(caption)
    print("Части на речта (POS Tags):")
    print(" | ".join(pos_tags))
    print("="*60)
    
    # Показваме самото изображение
    plt.figure(figsize=(4, 4))
    plt.imshow(image)
    plt.title(f"Label: {label}")
    plt.axis("off")
    plt.show()

### Решение

In [ ]:
from sklearn.decomposition import TruncatedSVD
import numpy as np

# Глобален SVD модел
_SVD_MODEL = None

def extract_features(dataset_split, batch_size=64, is_train=False):
    global _SVD_MODEL
    
    if is_train:
        # Използваме 29 компонента за SVD + 1(cos) + 2(NLP) = 32 Features
        _SVD_MODEL = TruncatedSVD(n_components=13, random_state=42)
        
    # --- Помощни функции ---
    def get_simple_nltk_feature(text: str) -> float:
        tokens = nltk.word_tokenize(text.lower())
        return len(set(tokens)) / len(tokens) if tokens else 0.0

    def get_simple_spacy_feature(text: str) -> float:
        doc = nlp(text)
        nouns = [t for t in doc if t.pos_ == "NOUN"]
        return len(nouns) / len(doc) if len(doc) > 0 else 0.0
    
    # --- Извличане ---
    X_raw = []
    all_diffs = []

    for i in tqdm(range(0, len(dataset_split), batch_size), desc="Извличане"):
        batch = dataset_split[i : i + batch_size]
        images = batch["image"]
        captions = batch["caption"]

        nltk_feats = [get_simple_nltk_feature(cap) for cap in captions]
        spacy_feats = [get_simple_spacy_feature(cap) for cap in captions]

        inputs = processor(text=captions, images=images, return_tensors="pt", padding=True).to(device)

        with torch.no_grad():
            outputs = model(**inputs)
            i_emb = outputs.image_embeds / outputs.image_embeds.norm(p=2, dim=-1, keepdim=True)
            t_emb = outputs.text_embeds / outputs.text_embeds.norm(p=2, dim=-1, keepdim=True)
            
            cos_sims = (i_emb * t_emb).sum(dim=-1).cpu().numpy()
            diffs = (i_emb - t_emb).cpu().numpy()

        for j in range(len(captions)):
            X_raw.append([cos_sims[j], nltk_feats[j], spacy_feats[j]])
            all_diffs.append(diffs[j])

    X_raw = np.array(X_raw)
    all_diffs = np.array(all_diffs)
    
    # Нормализация
    row_norms = np.linalg.norm(all_diffs, axis=1, keepdims=True)
    all_diffs = all_diffs / np.where(row_norms > 1e-8, row_norms, 1.0)
    all_diffs = all_diffs.astype(np.float64)
    

    # --- SVD Трансформация ---
    if is_train:
        svd_feats = _SVD_MODEL.fit_transform(all_diffs)
    else:
        svd_feats = _SVD_MODEL.transform(all_diffs)
        
    final_X = np.hstack([X_raw, svd_feats])
    
    # --- Стабилно скалиране ---
    mean = np.mean(final_X, axis=0)
    std = np.std(final_X, axis=0) + 1e-8
    return (final_X - mean) / std

### Оценяване

In [ ]:
# ==========================================
# ВНИМАНИЕ: АБСОЛЮТНО Е ЗАБРАНЕНО ДА ПРОМЕНЯТЕ ТОЗИ БЛОК!
# ==========================================
BATCH_SIZE = 64

print("--- Обработка на Train Split ---")
y_train = np.array(labeled_data["train"]["label"])
safe_train_data = labeled_data["train"].remove_columns("label")
X_train = extract_features(safe_train_data, batch_size=BATCH_SIZE, is_train=True)


print("\n--- Обработка на Val Split ---")
y_val = np.array(labeled_data["val"]["label"])
safe_val_data = labeled_data["val"].remove_columns("label")
X_val = extract_features(safe_val_data, batch_size=BATCH_SIZE)


# Проверка за позволения брой характеристики
assert X_train.shape[1] <= 16, f"Грешка: Използвате {X_train.shape[1]} характеристики, а лимитът е 16!"

print("\nОбучение на фиксирания Gradient Boosting Classifier...")

# Използваме пайплайн със StandardScaler за числена стабилност
clf = make_pipeline(
    StandardScaler(),
    GradientBoostingClassifier(
        n_estimators=400,
        max_depth=5,
        learning_rate=0.05,
        subsample=0.8,
        max_features="sqrt",
        min_samples_leaf=5,
        random_state=42,
    ),
)

# Тренираме модела
clf.fit(X_train, y_train)

# Предсказваме върху валидационния сет
y_val_pred = clf.predict(X_val)

# Изчисляване на водещите метрики
val_accuracy = accuracy_score(y_val, y_val_pred)
val_f1_macro = f1_score(y_val, y_val_pred, average="macro")

print("\n" + "=" * 50)
print(" (VALIDATION REPORT)")
print("=" * 50)
print(
    classification_report(
        y_val, y_val_pred, target_names=["0: Коректно", "1: Халюцинация", "2: Несвързано"]
    )
)
print("-" * 50)
print(f"Обща Точност (Accuracy): {val_accuracy:.4f}")
print(f"Водеща Метрика (Macro F1): {val_f1_macro:.4f}")
print("-" * 50)
print("\nМатрица на грешките (Confusion Matrix):")
print(confusion_matrix(y_val, y_val_pred))

--- Обработка на Train Split ---


Извличане: 100%|██████████| 38/38 [00:36<00:00,  1.03it/s]
/Users/delyan-boychev/.local/share/mamba/envs/ioai/lib/python3.10/site-packages/sklearn/utils/extmath.py:350: RuntimeWarning: divide by zero encountered in matmul
  Q, _ = normalizer(A @ Q)
/Users/delyan-boychev/.local/share/mamba/envs/ioai/lib/python3.10/site-packages/sklearn/utils/extmath.py:350: RuntimeWarning: overflow encountered in matmul
  Q, _ = normalizer(A @ Q)
/Users/delyan-boychev/.local/share/mamba/envs/ioai/lib/python3.10/site-packages/sklearn/utils/extmath.py:350: RuntimeWarning: invalid value encountered in matmul
  Q, _ = normalizer(A @ Q)
/Users/delyan-boychev/.local/share/mamba/envs/ioai/lib/python3.10/site-packages/sklearn/utils/extmath.py:351: RuntimeWarning: divide by zero encountered in matmul
  Q, _ = normalizer(A.T @ Q)
/Users/delyan-boychev/.local/share/mamba/envs/ioai/lib/python3.10/site-packages/sklearn/utils/extmath.py:351: RuntimeWarning: overflow encountered in matmul
  Q, _ = normalizer(A.T @ Q)


Shape: (2400, 512)
NaN: 0
Inf: 0
Min: -0.5706786513328552, Max: 0.30347320437431335
Нулеви редове: 0

--- Обработка на Val Split ---


Извличане: 100%|██████████| 10/10 [00:08<00:00,  1.12it/s]
/Users/delyan-boychev/.local/share/mamba/envs/ioai/lib/python3.10/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: divide by zero encountered in matmul
  ret = a @ b
/Users/delyan-boychev/.local/share/mamba/envs/ioai/lib/python3.10/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: overflow encountered in matmul
  ret = a @ b
/Users/delyan-boychev/.local/share/mamba/envs/ioai/lib/python3.10/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: invalid value encountered in matmul
  ret = a @ b


Shape: (600, 512)
NaN: 0
Inf: 0
Min: -0.5712284445762634, Max: 0.3124302327632904
Нулеви редове: 0

Обучение на фиксирания Gradient Boosting Classifier...

 (VALIDATION REPORT)
                precision    recall  f1-score   support

   0: Коректно       0.73      0.75      0.74       200
1: Халюцинация       0.75      0.73      0.74       200
 2: Несвързано       0.90      0.89      0.90       200

      accuracy                           0.79       600
     macro avg       0.79      0.79      0.79       600
  weighted avg       0.79      0.79      0.79       600

--------------------------------------------------
Обща Точност (Accuracy): 0.7917
Водеща Метрика (Macro F1): 0.7921
--------------------------------------------------

Матрица на грешките (Confusion Matrix):
[[150  42   8]
 [ 42 147  11]
 [ 14   8 178]]
